# Lab 7 · Chuỗi và biểu thức chính quy trên 690 nghìn đánh giá

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành · bài 7**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook demo Bài 7 xử lý cột `name` và `amenities` của bảng dữ liệu chỗ ở (`listings`). Lab này làm việc
với cột **comments** của bảng đánh giá đầy đủ (`reviews`), gồm 690.112 đoạn văn, để xây dựng
các tín hiệu văn bản sẽ được dùng lại ở Bài 11.

## Cách làm việc trong buổi lab

- Bài tập được chia thành từng bước; mỗi bước có ô `TODO` và phần kiểm tra `assert`.
  Hoàn thành toàn bộ `assert` nghĩa là kết quả đáp ứng yêu cầu.
- Phần khởi động và bài có hướng dẫn: bạn nên **tự gõ, không dùng AI** — các bài kiểm tra
  định kỳ 🚫 đóng ở giờ lý thuyết đánh giá các kỹ năng này.
- Bài tự làm ở cuối được gắn nhãn ✅ mở: bạn được dùng AI, kèm trách nhiệm khai báo
  theo chính sách AI của môn.
- Nếu chưa giải quyết được một bước sau 3 phút, hãy gọi giảng viên thực hành đến hỗ trợ.

## Mục tiêu

Sau buổi lab, bạn sẽ:

1. Khảo sát một cột văn bản lớn: độ dài, giá trị thiếu và nội dung ít thông tin.
2. Làm sạch HTML thừa bằng `str.replace` và kiểm chứng bằng đếm.
3. Viết quy tắc nhận diện ngôn ngữ và tín hiệu từ khoá bằng `str.contains`.
4. Dùng `str.extract` trên văn bản tự do và kiểm tra các trường hợp khớp nhầm.

## Phần 0 · Khởi động (~8 phút)

In [ ]:
import pandas as pd

# W1 — chuẩn hoá trước khi đếm
s = pd.Series(["  Wifi ", "wifi", "WIFI!", "wi-fi"])

# TODO: strip + lower cho cả cột
s_chuan = ...

# --- Ô kiểm tra ---
assert list(s_chuan) == ["wifi", "wifi", "wifi!", "wi-fi"]
assert (s_chuan == "wifi").sum() == 2
print("W1 ổn — chuẩn hoá gộp được 2 biến thể; 2 biến thể còn lại cần xử lý thêm.")

In [ ]:
# W2 — bẫy NaN của contains
t = pd.Series(["depto centro", None, "casa con vista"])

# TODO: đếm số dòng chứa "centro" sao cho dòng None được tính là KHÔNG chứa
so_chua = ...

# --- Ô kiểm tra ---
assert so_chua == 1
print("W2 ổn — na=False quy ước giá trị thiếu là 'không chứa'.")

## Phần 1 · Khảo sát cột văn bản lớn (~25 phút)

Bảng `reviews` **đầy đủ** có cột `comments` với 690.112 đoạn đánh giá bằng nhiều ngôn ngữ.

In [ ]:
URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/data/reviews.csv.gz")
rv = pd.read_csv(URL, usecols=["listing_id", "date", "comments"],
                 parse_dates=["date"])
len(rv)

### Bước 1 · Nhìn tổng quan cột chữ

Bắt đầu bằng ba bước: đếm giá trị thiếu → đo độ dài → kiểm tra một số mẫu ngắn và dài.

In [ ]:
# TODO: đếm giá trị thiếu trong cột comments; tạo c = cột đã bỏ NaN; tính độ dài từng đoạn
so_nan = ...
c = ...
do_dai = ...            # gợi ý: .str.len()

# --- Ô kiểm tra ---
assert so_nan == 36 and len(c) == 690076
assert do_dai.median() == 115.0
print(f"{so_nan} giá trị thiếu; độ dài trung vị {do_dai.median():.0f} ký tự.")

In [ ]:
# TODO: đếm số đoạn đánh giá ngắn hơn 20 ký tự, rồi xem 5 đoạn ngắn nhất
so_ngan = ...

# --- Ô kiểm tra ---
assert so_ngan == 58987
print(f"{so_ngan:,} đoạn đánh giá < 20 ký tự (~8,5%)")
c[do_dai < 3].head(5).tolist()

Gần 59 nghìn đoạn đánh giá ngắn hơn 20 ký tự; một số chỉ chứa dấu câu như "`.`".
Cần lọc các đoạn không mang đủ thông tin trước khi gửi tới LLM để tránh lãng phí hạn mức sử dụng.

### Bước 2 · Thẻ HTML `<br/>`

Airbnb lưu xuống dòng thành thẻ `<br/>` ngay trong văn bản.

In [ ]:
# TODO: đếm số đoạn đánh giá chứa "<br/>" (đặt regex=False), rồi tạo c_sach thay "<br/>" bằng " "
so_br = ...
c_sach = ...

# --- Ô kiểm tra ---
assert so_br == 116471
assert int(c_sach.str.contains("<br/>", regex=False).sum()) == 0
print(f"{so_br:,} đoạn đánh giá có <br/>; sau khi thay còn 0.")

## Phần 2 · Tín hiệu từ văn bản (~30 phút)

### Bước 3 · Quy tắc nhận diện ngôn ngữ

Bài giảng dùng một quy tắc xấp xỉ cho tiếng Tây Ban Nha trên cột `name`.
Ở đây, ta điều chỉnh bộ từ khoá và áp dụng cho `comments`:

In [ ]:
PAT_ES = r"ción|ñ|muy|excelente|departamento"

# TODO: tạo mặt nạ la_es (contains PAT_ES, không phân biệt hoa thường) và tính tỷ lệ
la_es = ...
ty_le_es = ...

# --- Ô kiểm tra ---
assert round(ty_le_es, 3) == 0.654
print(f"~{ty_le_es:.0%} đoạn đánh giá mang dấu hiệu tiếng Tây Ban Nha.")

In [ ]:
# TODO: so độ dài trung vị của nhóm es và nhóm còn lại
do_dai_sach = c_sach.str.len()
len_es = ...
len_khac = ...

# --- Ô kiểm tra ---
assert len_es == 118.0 and len_khac == 105.0
print(f"Trung vị độ dài — nhóm es: {len_es:.0f} · nhóm khác: {len_khac:.0f} ký tự.")

Quy tắc này chỉ là tín hiệu xấp xỉ: một đoạn đánh giá tiếng Anh chứa từ "departamento" vẫn
có thể bị gán vào nhóm `es`. Trong mẫu này, nhóm có tín hiệu tiếng Tây Ban Nha chiếm khoảng
hai phần ba và có độ dài trung vị cao hơn; Bài 11 sẽ đánh giá quy tắc bằng nhãn tay.

### Bước 4 · str.extract trên văn bản tự do và các trường hợp khớp nhầm

Đoạn đánh giá thường có cấu trúc "cerca de X" / "near X" (gần X). Ta trích **X** để xem
các địa điểm thường được nhắc đến:

In [ ]:
PAT_GAN = r"(?:cerca de la |cerca del |cerca de |near the |near )([A-Za-zÁ-Úá-úñÑ]+)"

# TODO: dùng str.extract với PAT_GAN (expand=False) trên c_sach; đếm số dòng trích được
gan = ...
so_trich = ...

# --- Ô kiểm tra ---
assert so_trich == 33841
gan.str.lower().value_counts().head(8)

Bảng tần suất có `metro` (7.640 lần), `centro`, `estación`, `mall` và cả `todo`, `muchos`.
Hai từ sau là trường hợp khớp nhầm: "cerca de todo" nghĩa là "gần mọi thứ", không phải
địa điểm. Biểu thức chính quy khớp cấu trúc nhưng không hiểu nghĩa; xem `value_counts()` của kết quả trích
xuất giúp phát hiện vấn đề này. `movistar` là một kết quả hợp lệ vì nhắc tới Movistar Arena.

## Phần 3 · Bài tự làm ✅ mở (làm sớm tại lớp hoặc làm tại nhà)

Bạn được dùng AI theo quy trình 5 bước; hãy ghi lại prompt và cách kiểm chứng.

### Tự làm 1 · Từ khoá hai ngôn ngữ

Xây 2 tín hiệu: `wifi` (mẫu `wifi|internet`) và `parking` (mẫu `parking|estacionamiento`).
Tính **tỷ lệ nhắc đến** của từng tín hiệu trong nhóm es và nhóm còn lại. Nhóm nào quan tâm
parking hơn? Nêu một cách giải thích có thể có và một giới hạn của cách giải thích đó.

### Tự làm 2 · Cải thiện mẫu "cerca de"

Sửa `PAT_GAN` để loại các trường hợp khớp nhầm như "todo"/"muchos" (gợi ý: thêm nhóm loại trừ
hoặc lọc kết quả sau khi trích bằng danh sách từ dừng). Đếm lại 8 giá trị phổ biến nhất — bảng mới sạch hơn
bảng cũ ở chỗ nào? Ghi lại quy trình: sửa mẫu → chạy lại → so kết quả (đúng vòng lặp
nới dần của bài giảng).

In [ ]:
# Viết bài tự làm của bạn ở đây

## Tóm tắt buổi lab

| Nội dung chính | Sẽ gặp lại ở |
|---|---|
| Khảo sát cột chữ: rỗng, độ dài, mẫu ngắn | lọc đoạn đánh giá trước khi gửi LLM (Bài 11) |
| Làm sạch `<br/>` và kiểm chứng bằng đếm | pipeline xử lý văn bản |
| Quy tắc ngôn ngữ trên 690.112 đoạn đánh giá | baseline cho phần LLM |
| `extract` và kiểm tra trường hợp khớp nhầm | kiểm chứng tín hiệu trích xuất |

Buổi lý thuyết tiếp theo: **dữ liệu thời gian** — dùng cột `date` để tổng hợp
và so sánh theo thời điểm.